<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/12-caching-memory/02-diagnosing-spill.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 12.2 — Diagnosing spill: find it, don't just report it

Force a spill with too few shuffle partitions for the data volume,
read `memoryBytesSpilled`/`diskBytesSpilled` off the Stages REST API,
find which task is actually responsible, then fix it two different
ways and confirm the spill drops.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName("12.2-diagnosing-spill")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
# local[*] note: one JVM plays driver + all executors, so the Executors tab
# shows a single "driver" entry — the memory MODEL below is identical to a
# real cluster, only the topology is collapsed.

## Stage-level spill via the REST API (extends 10.2's `recent_stages`)

In [ ]:
import urllib.request, json as _json

def spill_report(n=6):
    app_id = spark.sparkContext.applicationId
    url = f"http://localhost:4040/api/v1/applications/{app_id}/stages?status=complete"
    stages = _json.load(urllib.request.urlopen(url))
    for s in sorted(stages, key=lambda s: s["stageId"])[-n:]:
        mem = s.get("memoryBytesSpilled", 0)
        disk = s.get("diskBytesSpilled", 0)
        flag = "  <-- SPILL" if disk > 0 else ""
        print(f"stage {s['stageId']:>3}: mem_spill={mem:>14,}B  disk_spill={disk:>12,}B{flag}")

spill_report()

## Force a spill: huge sort, deliberately too few partitions

A wide sort over millions of rows with only 2 shuffle partitions
overflows each partition's execution-memory share.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "2")   # deliberately too few

wide = (spark.range(15_000_000)
        .withColumn("v", (col("id") * 2654435761) % 1000000007)
        .withColumn("pad", F.lit("y" * 300)))     # bulk rows up so 2 partitions really hurt

wide.orderBy("v").count()
print("spill with shuffle.partitions=2:")
spill_report(2)

## Find the responsible task, not just the stage total

One skewed/oversized task can carry almost all of a stage's spill.
Pull the task list for the last sort stage and sort by disk spill.

In [ ]:
def worst_tasks(stage_id, n=5):
    app_id = spark.sparkContext.applicationId
    url = (f"http://localhost:4040/api/v1/applications/{app_id}"
           f"/stages/{stage_id}")
    detail = _json.load(urllib.request.urlopen(url))[0]
    tasks = sorted(detail["tasks"].values(),
                    key=lambda t: t["taskMetrics"]["diskBytesSpilled"], reverse=True)
    for t in tasks[:n]:
        m = t["taskMetrics"]
        print(f"task {t['taskId']:>4}: disk_spill={m['diskBytesSpilled']:>12,}B "
              f"duration={t['duration']:>7}ms")

# find the sort stage's id from spill_report() output above, then:
app_id = spark.sparkContext.applicationId
stages = _json.load(urllib.request.urlopen(
    f"http://localhost:4040/api/v1/applications/{app_id}/stages?status=complete"))
sort_stage_id = max(s["stageId"] for s in stages if s.get("diskBytesSpilled", 0) > 0)
print(f"inspecting stage {sort_stage_id}:")
worst_tasks(sort_stage_id)

## Fix #1 — more shuffle partitions (10.3's rule, applied)

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "48")   # right-sized for this data

wide.orderBy("v").count()
print("spill with shuffle.partitions=48:")
spill_report(2)   # compare disk_spill to the run above — should shrink or vanish

## Fix #2 — more execution memory headroom (cheaper partitions, same effect)

Simulate "more headroom" by shrinking row size instead (stand-in for
raising executor memory, which local mode can't demonstrate directly).

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "2")  # back to the bad setting
narrow = spark.range(15_000_000).withColumn("v", (col("id") * 2654435761) % 1000000007)
# no wide 'pad' column this time -- much less bytes per task to buffer

narrow.orderBy("v").count()
print("same partition count, much smaller rows:")
spill_report(2)   # smaller rows per task -> less spill even with too few partitions

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "8")   # restore course default
spark.stop()

## Exercises

1. Re-run the "too few partitions" experiment with
   `shuffle.partitions` at 1, 2, 8, 48, 200. Plot (by hand or print) how
   disk spill changes. Where's the knee of the curve?
2. Build a skewed sort (one key 90% of rows, like 10.4) with a
   reasonable partition count and confirm most spill concentrates on
   one task via `worst_tasks()`. Does raising `shuffle.partitions`
   help this case as much as the evenly-distributed one? Why not?
3. Compare `memoryBytesSpilled` to `diskBytesSpilled` for the same
   stage — compute the compression ratio. Is it consistent across
   different partition counts?
4. Cache `wide` before sorting it, and re-check spill on the sort
   stage. Did caching change anything about the spill, or is spill
   orthogonal to whether the *input* was cached?
5. Write a one-cell "spill gate": raise an exception if any completed
   stage's `diskBytesSpilled` exceeds a threshold you pick. Test it
   against both the bad (`partitions=2`) and good (`partitions=48`)
   configurations.

In [ ]:
# your exercise code here